## IDEE 

In [18]:
class Layer:
    """classe che wrappa una funzione di transform.
    eg: i constrain sui parametri
    NOTE: I layer sono contenuti dentro una lista
    ITER no più RICORSIVO (migliori performance?)
    """

    def __init__(self, transform):
        self.transform = transform  # una funzione di trasformazione
        #self.next_layer = next_layer   

    def __call__(self, args, **kwargs):
        # transform(*args, **kwargs) dovrebbe restituire una tupla
        # (new_args, new_kwargs) o solo new_args se usi solo posizionali
        new_args, new_kwargs = self.transform(args, **kwargs)       
        return new_args, new_kwargs
    
    def __str__(self):
        # Proviamo a mostrare il nome della funzione se possibile, altrimenti il suo repr
        transform_name = getattr(self.transform, "__name__", repr(self.transform))
        return f"Layer(transform={transform_name})"

class LayerChain:
    """
    Questa classe funge da contenitore di layer (trasformazioni)
    e al tempo stesso da 'layer' unico, grazie al metodo __call__.
    """

    def __init__(self):
        # Nella forma più semplice memorizziamo i layer in una lista Python.
        self.layers = []

    def __call__(self, args, **kwargs):
        """
        Applica in sequenza tutte le trasformazioni contenute in self.layers.
        Infine, se esiste un next_layer, passa il risultato al next_layer.
        """
        for transform in self.layers:
            args, kwargs = transform(
                args, **kwargs
            )  # each transform deve restituire (args, kwargs)

        return args, kwargs

    def append(self, transform):
        """
        Aggiunge un layer in coda (ultimo) alla catena.
        """
        self.layers.append(transform)
        return self  # facoltativo, per chaining

    def prepend(self, transform):
        """
        Aggiunge un layer in testa (primo) alla catena.
        """
        self.layers.insert(0, transform)
        return self

    def insert_after(self, target_transform, new_transform):
        """
        Inserisce un nuovo layer subito dopo il primo layer `target_transform`.
        Se `target_transform` non esiste nella catena, solleva un ValueError.
        """
        for i, layer in enumerate(self.layers):
            if layer == target_transform:
                self.layers.insert(i + 1, new_transform)
                return self
        raise ValueError("target_transform non trovato nella catena.")
    
    def __str__(self):
        layers_str = ", ".join(str(layer) for layer in self.layers)
        return f"LayerChain(layers=[{layers_str}])"


class LayeredNode:
    """
    Classe base che fornisce la logica di "chiamare il layer" sugli argomenti in ingresso.
    """

    def __init__(self, layer=None):
        self.layer = layer

    def _apply_layer(self, args, **kwargs):
        """
        Se self.layer è impostato, trasformo gli argomenti con il layer.
        Restituisce sempre una tupla (args, kwargs).
        """
        if self.layer is not None:
            return self.layer(args, **kwargs)
        
        return args, kwargs
    
    def __str__(self):
        # Mostriamo semplicemente la presenza (o l'assenza) di un layer
        return f"LayeredNode(layer={self.layer})"

def componemodels(op):
    if op == '+':
        return lambda left, right: SumNode(left, right, layer=LayerChain())
    elif op == '-':
        return lambda left, right: DiffNode(left, right, layer=LayerChain())



class Node(LayeredNode):
    #NOTE posso creare nodi con logiche complicate e diminuire gli overheads
    # + --> SumNode
    # - --> DiffNode
    # * --> ProdNode e così via
    
    def __init__(self, left=None, right=None, layer=None):
        super().__init__(layer=layer)
        self.left = left
        self.right = right
    
    def evaluate(self, *args, **kwargs):
        raise NotImplementedError('Single Custom Nodes should implement combine logic')
    
    def __str__(self):
        left_str = str(self.left) if self.left else "None"
        right_str = str(self.right) if self.right else "None"
        layer_str = str(self.layer) if self.layer else "None"
        return (
            f"{self.__class__.__name__}(\n"
            f"  left={left_str},\n"
            f"  right={right_str},\n"
            f"  layer={layer_str}\n"
            f")"
        )
    
    __add__ = componemodels('+')
    __sub__ = componemodels('-')
    
    
    
    



class SumNode(Node):
    def __init__(self, left=None, right=None, layer=None):
        super().__init__(left, right, layer)

    def evaluate(self, *args, **kwargs):
        # args = list(args)
        # 1) Trasformazione degli argomenti in ingresso
        args, kwargs = self._apply_layer(args, **kwargs)

        # 2) Valutazione dei figli
        left_val = self.left.evaluate(*args, **kwargs)
        right_val = self.right.evaluate(*args, **kwargs)

        # 3) Combino i risultati (somma in questo esempio)
        return left_val + right_val

class DiffNode(Node):
    
    def __init__(self, left=None, right=None, layer=None):
        super().__init__(left, right, layer)

    def evaluate(self, *args, **kwargs):
        # args = list(args)
        # 1) Trasformazione degli argomenti in ingresso
        args, kwargs = self._apply_layer(args, **kwargs)

        # 2) Valutazione dei figli
        left_val = self.left.evaluate(*args, **kwargs)
        right_val = self.right.evaluate(*args, **kwargs)

        # 3) Combino i risultati (somma in questo esempio)
        return left_val - right_val


class Leaf(Node):
    def __init__(self, func, layer=None):
        super().__init__(left=None, right=None, layer=layer)
        self.func = func

    
    def evaluate(self, *args, **kwargs):   
        # args = list(args)
        args, kwargs = self._apply_layer(args, **kwargs)
        return self.func(*args, **kwargs)
    
    def __str__(self):
        func_name = getattr(self.func, "__name__", repr(self.func))
        layer_str = str(self.layer) if self.layer else "None"
        return (
            f"{self.__class__.__name__}(\n  func={func_name},\n  layer={layer_str}\n)"
        )


In [21]:
def double_first_arg(args, **kwargs):
    print(f"double got {args}, {kwargs}")
    #args[0] *= 2
    kwargs['x'] *=2
    return args, kwargs


def add_one_to_first_arg(args, **kwargs):
    print(f"add got {args}, {kwargs}")
    #args[0] += 1
    kwargs['x'] += 1
    return args, kwargs


def f1(x):
    print(f"f1({x})")
    return x + 10


def f2(x):
    print(f"f2({x})")
    return 5



# Esempio con un singolo layer
layer_for_leaf1 = Layer(double_first_arg)

# Esempio con una chain di più layer
chain_for_leaf2 = LayerChain()
chain_for_leaf2.append(double_first_arg)
chain_for_leaf2.append(add_one_to_first_arg)

# Foglie
leaf1 = Leaf(func=f1, layer=layer_for_leaf1)
leaf2 = Leaf(func=f2, layer=chain_for_leaf2)

# Nodo radice, anch'esso con un suo layer
root = leaf1 + leaf2
#root.layer.append(add_one_to_first_arg)
# Esecuzione
# NOTE per evitare la conversione Tuple -> List per ora funziona tutto solo con Kwargs
res = root.evaluate(x=1)
print("Risultato:", res)
print(root)


add got (), {'x': 1}
double got (), {'x': 2}
f1(4)
double got (), {'x': 2}
add got (), {'x': 4}
f2(5)
Risultato: 19
SumNode(
  left=Leaf(
  func=f1,
  layer=Layer(transform=double_first_arg)
),
  right=Leaf(
  func=f2,
  layer=LayerChain(layers=[<function double_first_arg at 0x7fcd7fc6c360>, <function add_one_to_first_arg at 0x7fcd7fc6c220>])
),
  layer=LayerChain(layers=[<function add_one_to_first_arg at 0x7fcd7fc6c220>])
)
